In [1]:
%matplotlib widget

In [2]:
%%capture
from ocelot import *

from ocelot.gui import *
import matplotlib.pyplot as plt
import numpy as np
import latdraw
from felarc.family import (get_isochronous_dispersions_at_middle_dipole_entrance, 
                           get_quad_strengths,
                           ArcDefinition, 
                           make_arc)
from felarc.t20 import ARC_ANGLE_RAD, ARC_ANGLE_DEG
from felarc.plot import plot_optics, plot_r56_full
from felarc.analysis import describe_arc_geometry, describe_arc_optics, get_curvature

The isochronous condition is given by $\int \frac{D(s)}{\rho(s)} \, \mathrm{d}s = 0$.
So in the most simple implementation: you offset the $\frac{D}{\rho}$ in the first and last dipoles with opposite-signed $\frac{D}{\rho}$ in the middle dipole.  The only way to do this is with quadrupoles inbetween to focus the dispersive trajectory so that its sign is flipped. 

# Introduction
Let's start with defining a simple isochronous arc.


$D_j = \rho_2\left [ D_j' \cot{(\phi_2) + 1} \right ]$

$D'_j = -\frac{\rho_1}{\rho_2} \left ( \frac{3}{2} \phi_1 - \sin\phi_1 \right )$

In [3]:
# MAGNET_ANGLE_RAD = 0.01 * 4
MAGNET_ANGLE_RAD = ARC_ANGLE_RAD / 3
MAGNET_ANGLE_DEG = ARC_ANGLE_DEG / 3
MAGNET_LENGTH = 3
dx, dxp = get_isochronous_dispersions_at_middle_dipole_entrance(MAGNET_ANGLE_RAD, MAGNET_LENGTH,
                                                                MAGNET_ANGLE_RAD, MAGNET_LENGTH)
print(f"Consider an arc consisting of three identical dipoles, L = {MAGNET_LENGTH}m, ANGLE = {MAGNET_ANGLE_RAD}rad = {MAGNET_ANGLE_DEG}")
print(f"To achieve an isochronous arc we need at the entrance to the middle dipole, DX = {dx:3g}m and DX' = {dxp:3g}rad")

Consider an arc consisting of three identical dipoles, L = 3m, ANGLE = 0.03490658503988659rad = 2.0
To achieve an isochronous arc we need at the entrance to the middle dipole, DX = -0.0261741m and DX' = -0.0174604rad


So we need to introduce quadrupoles to achieve the necessary dispersions at the entrance to the middle dipole.  

## Defining an Arc

In [4]:
# Define a simple symmetrical arc consisting of three dipoles, 4 quads, and six drifts.
bd_arc = ArcDefinition(MAGNET_ANGLE_RAD,
                       MAGNET_LENGTH,
                       MAGNET_ANGLE_RAD,
                       MAGNET_LENGTH,
                       l1=0.5,
                       l2=2,
                       l3=2,
                       quad_length=1,
                       k1_outer=1.0,
                       k1_inner=-0.1
                       )
# Get the quadrupole strengths that give us the isochronous condition and print them
bd_arc = get_quad_strengths(bd_arc)
print(f"{bd_arc.k1_outer = :.4g}, {bd_arc.k1_inner = :.4g}")

Optimization terminated successfully.
         Current function value: 0.000000
         Iterations: 131
         Function evaluations: 240


ValueError: too many values to unpack (expected 2)

In [ ]:
# Define a lattice that goes up to only the start of the middle dipole. 
# We do this to check we have the dispersion we expect.
tws0 = Twiss(beta_x = 23, alpha_x = 3, beta_y=5.3, alpha_y=-0.28, E=14)
mlat_half = make_arc_to_inner_dipole_start(bd_arc)
describe_arc_optics(tws0, mlat_half)

In [ ]:
# We saw above that we have the expected Dx and Dx'.
# Now we build the full arc and check the optics
mlat_full = make_arc(bd_arc)
describe_arc_optics(tws0, mlat_full)

# Comment
And so we can see that this arc is isochronous, but the transverse beta functions are unreasonably large.

Now to look at the evolution of this integral $\frac{D}{\rho}$.

In [ ]:
plot_r56_full(tws0, mlat_full)
plt.show()

In [ ]:
describe_arc_geometry(mlat_full.sequence)

In [ ]:
plot_optics(tws0, mlat_full);

As can be seen, whilst the lattice is indeed isochronous, the transverse optics functions are unacceptably large.  So we need to introduce a waist in the central dipole (i.e. $\alpha_x = \alpha_y = 0.0$).